# Task 2.3 — Result, Comparison and Reproducibility Checklist
**Paper:** One-Sided Support Vector Regression for Multiclass Cost-Sensitive Classification (Tu & Lin, ICML 2010)  
**Student:** Vaishnav Verma (230160)

In [10]:
# === Configuration & Seeds (same as Task 2.2) ===
import numpy as np
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

TEST_SIZE = 0.3
C_REG = 100.0
GAMMA = 0.5

COST_MATRIX = np.array([
    [0, 1, 5],
    [2, 0, 3],
    [10, 1, 0],
])

In [11]:
# === Reproduce Setup from Task 2.2 ===
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics.pairwise import rbf_kernel
from scipy.optimize import minimize

wine = load_wine()
X = wine.data
y = wine.target
K = len(np.unique(y))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

# Regret vectors
regret_train = COST_MATRIX[y_train]
regret_test = COST_MATRIX[y_test]

print(f"Data loaded: {X_train.shape[0]} train, {X_test.shape[0]} test")

Data loaded: 124 train, 54 test


In [12]:
# === OSSVR Implementation (same as Task 2.2 — dual QP, no bias) ===
import cvxpy as cp

class OneSidedSVR:
    def __init__(self, C=1.0, gamma='scale'):
        self.C = C
        self.gamma = gamma
        self.alpha_ = None
        self.X_train_ = None
        self.gamma_value_ = None
    
    def _compute_gamma(self, X):
        if self.gamma == 'scale':
            return 1.0 / (X.shape[1] * X.var())
        return self.gamma
    
    def fit(self, X, r):
        N = X.shape[0]
        self.X_train_ = X.copy()
        self.gamma_value_ = self._compute_gamma(X)
        K_mat = rbf_kernel(X, X, gamma=self.gamma_value_)
        K_mat = (K_mat + K_mat.T) / 2
        K_mat += 1e-8 * np.eye(N)
        
        alpha = cp.Variable(N)
        objective = cp.Maximize(r @ alpha - 0.5 * cp.quad_form(alpha, K_mat))
        constraints = [alpha >= 0, alpha <= self.C]
        prob = cp.Problem(objective, constraints)
        prob.solve(solver=cp.CLARABEL, verbose=False)
        self.alpha_ = np.array(alpha.value).flatten()
        return self
    
    def predict(self, X):
        K_mat = rbf_kernel(X, self.X_train_, gamma=self.gamma_value_)
        return K_mat @ self.alpha_

def ossvr_predict(models, X):
    predictions = np.column_stack([m.predict(X) for m in models])
    return np.argmin(predictions, axis=1)

def avg_misclassification_cost(y_true, y_pred, cost_matrix):
    return np.mean(cost_matrix[y_true, y_pred])

# Train OSSVR
ossvr_models = []
for k in range(K):
    model = OneSidedSVR(C=C_REG, gamma=GAMMA)
    model.fit(X_train, regret_train[:, k])
    ossvr_models.append(model)

y_pred_test = ossvr_predict(ossvr_models, X_test)
ossvr_cost = avg_misclassification_cost(y_test, y_pred_test, COST_MATRIX)
ossvr_acc = accuracy_score(y_test, y_pred_test)

print(f"OSSVR — Test Avg Cost: {ossvr_cost:.4f}, Accuracy: {ossvr_acc:.4f}")

OSSVR — Test Avg Cost: 0.0741, Accuracy: 0.9444


In [13]:
# === Baseline: Cost-Sensitive OVA SVM (CS-OVA) ===
# This is the main baseline from the paper.
# Uses sklearn SVC with class_weight to encode costs.
from sklearn.svm import SVC

# Derive class_weight from cost matrix: average off-diagonal cost per true class
class_weights = {}
for k in range(K):
    off_diag = [COST_MATRIX[k, j] for j in range(K) if j != k]
    class_weights[k] = np.mean(off_diag)

print(f"CS-OVA class weights: {class_weights}")

cs_ova = SVC(C=C_REG, kernel='rbf', gamma=GAMMA, 
             class_weight=class_weights, random_state=RANDOM_SEED)
cs_ova.fit(X_train, y_train)
y_pred_csova = cs_ova.predict(X_test)

csova_cost = avg_misclassification_cost(y_test, y_pred_csova, COST_MATRIX)
csova_acc = accuracy_score(y_test, y_pred_csova)

print(f"CS-OVA — Test Avg Cost: {csova_cost:.4f}, Accuracy: {csova_acc:.4f}")

CS-OVA class weights: {0: np.float64(3.0), 1: np.float64(2.5), 2: np.float64(5.5)}
CS-OVA — Test Avg Cost: 0.1296, Accuracy: 0.8704


In [14]:
# === Standard SVM Baseline (cost-unaware) ===
std_svm = SVC(C=C_REG, kernel='rbf', gamma=GAMMA, random_state=RANDOM_SEED)
std_svm.fit(X_train, y_train)
y_pred_std = std_svm.predict(X_test)

std_cost = avg_misclassification_cost(y_test, y_pred_std, COST_MATRIX)
std_acc = accuracy_score(y_test, y_pred_std)

print(f"Std SVM — Test Avg Cost: {std_cost:.4f}, Accuracy: {std_acc:.4f}")

Std SVM — Test Avg Cost: 0.1296, Accuracy: 0.8704


## Result Comparison

In [15]:
# === Results Table ===
import pandas as pd

results = pd.DataFrame({
    'Method': ['OSSVR (Ours)', 'CS-OVA SVM (Baseline)', 'Standard SVM'],
    'Avg Misclassification Cost': [ossvr_cost, csova_cost, std_cost],
    'Accuracy': [ossvr_acc, csova_acc, std_acc],
})

print("=" * 65)
print("RESULTS COMPARISON")
print("=" * 65)
print(results.to_string(index=False))
print("=" * 65)

RESULTS COMPARISON
               Method  Avg Misclassification Cost  Accuracy
         OSSVR (Ours)                    0.074074  0.944444
CS-OVA SVM (Baseline)                    0.129630  0.870370
         Standard SVM                    0.129630  0.870370


## Discussion of Results

The results above compare OSSVR against the two baselines on average misclassification cost (lower is better) and accuracy. The paper reports that OSSVR consistently achieves lower cost than CS-OVA and standard SVM on UCI datasets. Our reproduction on the Wine dataset with a custom cost matrix may show similar trends, though the exact numbers differ from the paper for several reasons:

1. **Different cost matrix:** The paper uses cost matrices from established benchmarks; ours is manually designed for illustration.
2. **Simplified solver:** We use L-BFGS-B on the primal instead of a dedicated QP solver on the dual, which may lead to slightly different solutions.
3. **Small dataset:** With only 54 test examples, results are sensitive to the specific train/test split.
4. **No hyperparameter tuning:** We use default C=1.0 and gamma='scale' without cross-validation, whereas the paper tunes hyperparameters.

These differences are expected and do not indicate a failure of the method. The key observation is whether OSSVR achieves lower cost than the baselines on this non-uniform cost matrix.

In [16]:
# === Visualisation: Cost-Weighted Confusion Matrix + Bar Chart ===
import os
# Robust path: works whether CWD is workspace root or partB/
_cwd = os.path.abspath('')
RESULTS_DIR = os.path.join(_cwd, 'results') if _cwd.endswith('partB') else os.path.join(_cwd, 'partB', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Confusion Matrices ---
methods = [
    ('OSSVR', y_pred_test),
    ('CS-OVA SVM', y_pred_csova),
    ('Standard SVM', y_pred_std),
]

for ax, (name, y_pred) in zip(axes[:2], methods[:2]):
    cm = confusion_matrix(y_test, y_pred)
    # Cost-weighted confusion: multiply counts by cost
    cost_cm = cm * COST_MATRIX
    sns.heatmap(cost_cm, annot=True, fmt='d', cmap='Reds', ax=ax,
                xticklabels=[f'Pred {i}' for i in range(K)],
                yticklabels=[f'True {i}' for i in range(K)])
    total_cost = cost_cm.sum()
    ax.set_title(f'{name}\nTotal Cost: {total_cost}')
    ax.set_ylabel('True Class')
    ax.set_xlabel('Predicted Class')

# --- Bar Chart ---
ax = axes[2]
method_names = [m[0] for m in methods]
costs = [avg_misclassification_cost(y_test, m[1], COST_MATRIX) for m in methods]
colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(method_names, costs, color=colors, edgecolor='black')
ax.set_ylabel('Average Misclassification Cost')
ax.set_title('Cost Comparison')
for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{cost:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
save_path = os.path.join(RESULTS_DIR, 'task_2_3_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved to {save_path}")

Figure saved to /Users/vaishnavverma/Downloads/230160-midsem/partB/results/task_2_3_comparison.png


/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_77575/2323760413.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The figure above shows cost-weighted confusion matrices for OSSVR and CS-OVA, plus a bar chart comparing average misclassification cost across all three methods. The cost-weighted confusion matrix highlights which specific misclassification errors contribute most to the total cost.

## Reproducibility Checklist

- [x] **Random seeds are set and documented** at the top of each notebook: `RANDOM_SEED = 42` is used for `numpy`, `random`, and all sklearn estimators.
- [x] **All dependencies are listed in `requirements.txt`** with version numbers: numpy, scipy, scikit-learn, matplotlib, seaborn, pandas, cvxpy, ipykernel.
- [x] **All notebooks run from top to bottom** in a clean environment without errors. Each notebook is self-contained.
- [x] **Dataset loading requires no undocumented manual steps.** The Wine dataset is loaded directly via `sklearn.datasets.load_wine()` — no file downloads needed.
- [x] **All hyperparameters are clearly named and defined in one place** at the top of each notebook: `C_REG`, `GAMMA`, `EPSILON`, `TEST_SIZE`, `COST_MATRIX`.